# Trace Analysis

Workflow:
1. **Scan** — load results, find failures, spot patterns
2. **Read** — pick a trace, read the conversation step by step
3. **Note** — write down what you observe (open coding)
4. **Cluster** — after 20-30 traces, group your notes into failure categories
5. **Fix** — change prompts/tools/behavior, rerun, compare

In [1]:
import json
from pathlib import Path

import pandas as pd

TRACES_DIR = Path("../traces")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 50)

## 1. Scan — find failures and patterns

In [2]:
results = pd.read_csv(TRACES_DIR / "results.csv")
results["passed"] = results["passed"].astype(bool)

print(f"{len(results)} results: {results['passed'].sum()} passed, {(~results['passed']).sum()} failed")

display_cols = ["workspace", "question", "passed", "tool_calls", "failed_assertions"]
if "category" in results.columns:
    display_cols.insert(1, "category")
results[display_cols].sort_values("passed")

28 results: 21 passed, 7 failed


,workspace,question,passed,tool_calls,failed_assertions
0,federalist-papers,What are the main themes across the Federalist Papers?,False,0,min_tool_calls >= 3; has_answer
25,sherlock-holmes,What methods does Holmes use across the stories?,False,20,has_answer
20,origin-of-species,What does Darwin say about natural selection in Chapter 4?,False,20,contains 'natural selection'; has_answer
18,federalist-papers,How do Hamilton and Madison differ on federal power?,False,20,contains 'hamilton'; contains 'madison'; has_answer
16,federalist-papers,What are the main themes across the Federalist Papers?,False,20,has_answer
8,world-factbook,What is the population of Japan?,False,0,"contains_any ['123,201,945', '123.2 million', '123 milli..."
27,world-factbook,Compare the economies of Brazil and Argentina.,False,20,contains 'brazil'; contains 'argentina'; has_answer
7,federalist-papers,How do Hamilton and Madison differ on federal power?,True,12,NaN
1,federalist-papers,What does Federalist No. 10 argue about factions?,True,4,NaN
24,sherlock-holmes,How does Holmes solve the case in The Speckled Band?,True,13,NaN


In [3]:
# Pass rate by category (if available)
if "category" in results.columns:
    results.groupby("category")["passed"].agg(["sum", "count", "mean"]).rename(
        columns={"sum": "passed", "count": "total", "mean": "pass_rate"}
    ).sort_values("pass_rate")
else:
    print("No category column — rerun collect_traces.py to populate it")

No category column — rerun collect_traces.py to populate it


In [4]:
# Pass rate by model
results.groupby("model")["passed"].agg(["sum", "count", "mean"]).rename(
    columns={"sum": "passed", "count": "total", "mean": "pass_rate"}
).sort_values("pass_rate")

,passed,total,pass_rate
model,,,
openrouter/deepseek/deepseek-v3.2,9,14,0.642857
openrouter/qwen/qwen3-coder,12,14,0.857143


In [5]:
# Spot the two failure modes: 0 tools = skipped tools, 20 tools = hit max_turns
failures = results[~results["passed"]]
failures[["model", "workspace", "question", "tool_calls", "failed_assertions", "error"]]

,model,workspace,question,tool_calls,failed_assertions,error
0,openrouter/qwen/qwen3-coder,federalist-papers,What are the main themes across the Federalist Papers?,0,min_tool_calls >= 3; has_answer,"litellm.APIError: APIError: OpenrouterException - {""erro..."
8,openrouter/qwen/qwen3-coder,world-factbook,What is the population of Japan?,0,"contains_any ['123,201,945', '123.2 million', '123 milli...",NaN
16,openrouter/deepseek/deepseek-v3.2,federalist-papers,What are the main themes across the Federalist Papers?,20,has_answer,NaN
18,openrouter/deepseek/deepseek-v3.2,federalist-papers,How do Hamilton and Madison differ on federal power?,20,contains 'hamilton'; contains 'madison'; has_answer,NaN
20,openrouter/deepseek/deepseek-v3.2,origin-of-species,What does Darwin say about natural selection in Chapter 4?,20,contains 'natural selection'; has_answer,NaN
25,openrouter/deepseek/deepseek-v3.2,sherlock-holmes,What methods does Holmes use across the stories?,20,has_answer,NaN
27,openrouter/deepseek/deepseek-v3.2,world-factbook,Compare the economies of Brazil and Argentina.,20,contains 'brazil'; contains 'argentina'; has_answer,NaN


## 2. Read — pick a trace, step through the conversation

In [ ]:
def load_trace(model_slug: str, workspace: str, question_slug: str) -> dict:
    path = TRACES_DIR / model_slug / workspace / f"{question_slug}.json"
    return json.loads(path.read_text())


def list_traces(model_slug: str) -> pd.DataFrame:
    """List all available trace files for a model."""
    rows = []
    model_dir = TRACES_DIR / model_slug
    for ws_dir in sorted(model_dir.iterdir()):
        if not ws_dir.is_dir():
            continue
        for trace_file in sorted(ws_dir.glob("*.json")):
            data = json.loads(trace_file.read_text())
            inner = data.get("trace", data)
            rows.append({
                "workspace": ws_dir.name,
                "slug": trace_file.stem,
                "passed": data["passed"],
                "tools": len(inner.get("tool_calls", [])),
                "question": data["question"][:60],
            })
    return pd.DataFrame(rows)


# List available models
print("Available models:")
for d in sorted(TRACES_DIR.iterdir()):
    if d.is_dir():
        print(f"  {d.name}")

In [7]:
# Pick a model and list its traces
MODEL = "openrouter-qwen-qwen3-coder"  # <-- change this
list_traces(MODEL)

,workspace,slug,passed,tools,question
0,federalist-papers,how-do-hamilton-and-madison-differ-on-federal-power,True,12,How do Hamilton and Madison differ on federal power?
1,federalist-papers,what-are-the-main-themes-across-the-federalist-papers,False,0,What are the main themes across the Federalist Papers?
2,federalist-papers,what-does-federalist-no-10-argue-about-factions,True,4,What does Federalist No. 10 argue about factions?
3,federalist-papers,what-does-hamilton-say-about-standing-armies,True,7,What does Hamilton say about standing armies?
4,federalist-papers,which-papers-discuss-the-judiciary,True,5,Which papers discuss the judiciary?
5,origin-of-species,how-does-darwin-explain-the-struggle-for-existence,True,4,How does Darwin explain the struggle for existence?
6,origin-of-species,what-does-darwin-say-about-natural-selection-in-chapter-4,True,2,What does Darwin say about natural selection in Chapter 4?
7,origin-of-species,what-examples-of-variation-under-domestication-does-darw...,True,8,What examples of variation under domestication does Darw...
8,sherlock-holmes,how-does-holmes-solve-the-case-in-the-speckled-band,True,6,How does Holmes solve the case in The Speckled Band?
9,sherlock-holmes,what-happens-in-a-scandal-in-bohemia,True,2,"What happens in ""A Scandal in Bohemia""?"


In [ ]:
def show_trace(data: dict) -> None:
    """Print a trace as a readable conversation."""
    inner = data.get("trace", data)
    tool_calls = inner.get("tool_calls", [])

    print(f"Question: {data['question']}")
    print(f"Passed: {data['passed']} | Tools: {len(tool_calls)}")
    if inner.get("error"):
        print(f"Error: {inner['error']}")
    print(f"Assertions: {data.get('assertions', {})}")
    print("=" * 80)

    for i, tc in enumerate(tool_calls, 1):
        args = json.loads(tc["arguments"])
        print(f"\n--- Step {i}: {tc['name']} ---")
        for k, v in args.items():
            print(f"  {k}: {v}")
        if tc.get("error"):
            print(f"  ERROR: {tc['error']}")
        elif tc.get("hit") is not None:
            print(f"  hit: {tc['hit']} | result_count: {tc.get('result_count')}")

    print("\n" + "=" * 80)
    print("ANSWER:")
    print(inner.get("answer") or "(no answer — hit max_turns)")


# Load and read a trace — change the workspace and slug
trace = load_trace(MODEL, "federalist-papers", "which-papers-discuss-the-judiciary")
show_trace(trace)

### Full API view — see exactly what the model received

For traces collected with the latest `collect_traces.py`, the full messages
array is stored in the trace JSON. For older traces, this field will be missing.

In [ ]:
LOGS_PATH = Path("../logs/llm_calls.jsonl")


def load_api_calls() -> list[dict]:
    """Load all raw API calls from the JSONL log."""
    if not LOGS_PATH.exists():
        print(f"No log file at {LOGS_PATH}")
        return []
    with LOGS_PATH.open() as f:
        return [json.loads(line) for line in f]


def find_conversation(api_calls: list[dict], question: str) -> list[dict]:
    """Find the sequence of API calls for a given question.

    Looks for calls where the user message matches, then collects
    consecutive calls with growing message counts (same conversation).
    """
    matches = []
    for i, call in enumerate(api_calls):
        msgs = call.get("messages", [])
        user_msgs = [m for m in msgs if m["role"] == "user"]
        if user_msgs and question.lower() in user_msgs[-1].get("content", "").lower():
            matches.append(i)

    if not matches:
        print(f"No API calls found matching: {question[:60]}")
        return []

    # Take the last match (most recent run) and collect the full conversation
    start = matches[-1]
    conversation = [api_calls[start]]
    prev_msg_count = len(api_calls[start]["messages"])

    for i in range(start + 1, len(api_calls)):
        msg_count = len(api_calls[i]["messages"])
        if msg_count > prev_msg_count:
            conversation.append(api_calls[i])
            prev_msg_count = msg_count
        else:
            break

    return conversation


def show_api_turn(call: dict, turn: int, show_content: bool = True, max_chars: int | None = None) -> None:
    """Print one API call — what the model saw and what it responded."""
    msgs = call["messages"]
    prompt_tokens = call.get("prompt_tokens", "?")
    completion_tokens = call.get("completion_tokens", "?")

    def truncate(text: str) -> str:
        if max_chars is None or len(text) <= max_chars:
            return text
        return text[:max_chars] + f"\n  ... ({len(text) - max_chars} more chars)"

    print(f"{'=' * 80}")
    print(f"TURN {turn} | {len(msgs)} messages | prompt: {prompt_tokens} tokens | completion: {completion_tokens} tokens")
    print(f"{'=' * 80}")

    for m in msgs:
        role = m["role"]
        content = m.get("content") or ""
        tool_calls = m.get("tool_calls", [])

        if role == "system":
            print(f"\n[system] ({len(content)} chars)")
            if show_content:
                print(truncate(content))
        elif role == "user":
            print(f"\n[user] {content}")
        elif role == "assistant":
            if tool_calls:
                for tc in tool_calls:
                    fn = tc["function"]
                    print(f"\n[assistant] calls {fn['name']}({fn['arguments'][:200]})")
            elif content:
                print(f"\n[assistant] ({len(content)} chars)")
                if show_content:
                    print(truncate(content))
        elif role == "tool":
            print(f"\n[tool result] ({len(content)} chars)")
            if show_content:
                print(truncate(content))

    # Show what the model responded
    resp = call.get("response")
    if resp:
        print(f"\n--- RESPONSE ---")
        if resp.get("tool_calls"):
            for tc in resp["tool_calls"]:
                print(f"  calls {tc['name']}({tc['arguments'][:200]})")
        elif resp.get("content"):
            print(f"  {truncate(resp['content'])}")


api_calls = load_api_calls()
print(f"Loaded {len(api_calls)} API calls from {LOGS_PATH}")

In [ ]:
def show_messages(messages: list[dict], max_chars: int | None = None) -> None:
    """Print the full message array from a trace."""
    def truncate(text: str) -> str:
        if max_chars is None or len(text) <= max_chars:
            return text
        return text[:max_chars] + f"\n  ... ({len(text) - max_chars} more chars)"

    for m in messages:
        role = m["role"]
        content = m.get("content") or ""
        tool_calls = m.get("tool_calls", [])

        if role == "system":
            print(f"\n{'=' * 80}")
            print(f"[system] ({len(content)} chars)")
            print(truncate(content))
        elif role == "user":
            print(f"\n{'=' * 80}")
            print(f"[user] {content}")
        elif role == "assistant":
            if tool_calls:
                for tc in tool_calls:
                    fn = tc["function"]
                    print(f"\n[assistant] calls {fn['name']}({fn['arguments'][:200]})")
            elif content:
                print(f"\n{'=' * 80}")
                print(f"[assistant] ({len(content)} chars)")
                print(truncate(content))
        elif role == "tool":
            print(f"\n[tool result] ({len(content)} chars)")
            print(truncate(content))


# Show full conversation from a trace (requires traces from latest collect_traces.py)
trace = load_trace(MODEL, "federalist-papers", "which-papers-discuss-the-judiciary")
inner = trace.get("trace", trace)

if inner.get("messages"):
    show_messages(inner["messages"])
else:
    print("No messages in this trace — rerun collect_traces.py to populate")

## 3. Note — open coding

After reading each trace, write a short note about what you observed.
After 20-30 traces, patterns will emerge. Add notes as rows below.

In [11]:
# Add a row each time you read a trace.
# After 20-30, look for clusters.
notes = pd.DataFrame([
    # {"model": MODEL, "workspace": "federalist-papers", "slug": "which-papers-discuss-the-judiciary", "note": "paginated read_file in chunks of 100 when whole file would fit"},
    # {"model": MODEL, "workspace": "world-factbook", "slug": "what-is-the-population-of-japan", "note": "answered from memory, 0 tool calls"},
    {"model": MODEL, "workspace": "federalist-papers", "slug": "which-papers-discuss-the-judiciary", "note": "model infers papers from file names without reading them. it just verified a few by reading the first 20 lines. Then it became confident enough to answer from its priors."}
], columns=["model", "workspace", "slug", "note"])
notes

,model,workspace,slug,note
0,openrouter-qwen-qwen3-coder,federalist-papers,which-papers-discuss-the-judiciary,model infers papers from file names without reading them...


## 4. Cluster — group notes into failure categories

Once you have enough notes, group them. Common categories:
- **skipped tools** — answered from memory
- **over-searching** — hit max_turns without concluding
- **pagination waste** — read_file in small chunks instead of whole file
- **wrong grounding** — found evidence but overrode it with training data
- **missed variant** — searched one term, didn't try synonyms

In [12]:
# Once you've filled in notes above, count them by pattern
if len(notes) > 0:
    notes["note"].value_counts()
else:
    print("No notes yet — start reading traces!")

## 5. Compare — tool call patterns across traces

In [ ]:
def trace_summary(model_slug: str) -> pd.DataFrame:
    """Load all traces for a model and summarize tool usage."""
    rows = []
    model_dir = TRACES_DIR / model_slug
    for ws_dir in sorted(model_dir.iterdir()):
        if not ws_dir.is_dir():
            continue
        for trace_file in sorted(ws_dir.glob("*.json")):
            data = json.loads(trace_file.read_text())
            inner = data.get("trace", data)
            tool_names = [tc["name"] for tc in inner.get("tool_calls", [])]
            abbreviations = {"list_files": "L", "search_files": "S", "read_file": "R",
                             "run_python": "P", "calculator": "C", "get_current_time": "T"}
            sequence = "→".join(abbreviations.get(t, "?") for t in tool_names)
            rows.append({
                "workspace": ws_dir.name,
                "question": data["question"][:50],
                "passed": data["passed"],
                "tools": len(tool_names),
                "sequence": sequence,
                "searches": tool_names.count("search_files"),
                "reads": tool_names.count("read_file"),
                "lists": tool_names.count("list_files"),
            })
    return pd.DataFrame(rows)

trace_summary(MODEL)

In [14]:
# Tool calls CSV — if available (requires a collect_traces run with the latest schema)
tool_calls_csv = TRACES_DIR / "tool_calls.csv"
if tool_calls_csv.exists():
    tc_df = pd.read_csv(tool_calls_csv)
    print(f"{len(tc_df)} tool calls across {tc_df['trace_id'].nunique()} traces")

    # Error rate by tool
    tc_df["has_error"] = tc_df["error"].notna() & (tc_df["error"] != "")
    tc_df.groupby("tool")["has_error"].agg(["sum", "count", "mean"]).rename(
        columns={"sum": "errors", "count": "total", "mean": "error_rate"}
    ).sort_values("error_rate", ascending=False)
else:
    print("No tool_calls.csv yet — run: uv run python scripts/collect_traces.py")

No tool_calls.csv yet — run: uv run python scripts/collect_traces.py
